In [70]:

import pandas as pd
import os

# Universal file loader function
def load_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == '.csv':
        df = pd.read_csv(file_path)
    elif ext in ['.xls', '.xlsx']:
        df = pd.read_excel(file_path, engine='openpyxl')
    elif ext == '.json':
        df = pd.read_json(file_path)
    elif ext == '.parquet':
        df = pd.read_parquet(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    print(f" Loaded file: {file_path}")
    print(f"Shape: {df.shape}")
    print("\nDynamic Attribute Summary:")
    for col in df.columns:
        dtype = df[col].dtype
        sample_values = df[col].dropna().head(3).tolist()
        print(f"Column: {col} | Type: {dtype} | Sample: {sample_values}")

    return df

# Example usage
file_path = r"synthetic_data.csv"  # Replace with your file path
df = load_file(file_path)  # df now holds your dataset


 Loaded file: synthetic_data.csv
Shape: (100000, 77)

Dynamic Attribute Summary:
Column: member_id | Type: object | Sample: ['MEMJTPNPBKRBD', 'MEM2UCIN3S40P', 'MEMUP3B47V9G1']
Column: encounter_id | Type: object | Sample: ['ENC202308035284', 'ENC202204298091', 'ENC202303201658']
Column: age | Type: int64 | Sample: [63, 23, 66]
Column: sex | Type: object | Sample: ['Female', 'Female', 'Male']
Column: weight | Type: float64 | Sample: [105.4, 78.8, 90.6]
Column: bmi | Type: float64 | Sample: [34.7, 27.5, 28.2]
Column: smoker | Type: bool | Sample: [False, False, False]
Column: region | Type: object | Sample: ['Northeast', 'Midwest', 'Northwest']
Column: race | Type: object | Sample: ['White', 'AfricanAmerican', 'White']
Column: provider_id | Type: object | Sample: ['PROV0166', 'PROV0185', 'PROV0163']
Column: npi_number | Type: int64 | Sample: [9068671239, 6895786277, 5176466344]
Column: speciality | Type: object | Sample: ['Internal Medicine', 'Orthopedics', 'Orthopedics']
Column: provide

In [71]:
df.isnull().sum()

member_id           0
encounter_id        0
age                 0
sex                 0
weight              0
                ...  
readmitted          0
claim_id            0
diag_1              0
diag_2          33043
diag_3          59874
Length: 77, dtype: int64

In [72]:
import hashlib
# Helper function for hashing IDs
def hash_id(x):
    return hashlib.sha256(str(x).encode()).hexdigest()


In [73]:

# Create dim_patient from original dataset
dim_patient = df[['member_id', 'age', 'sex', 'region']]

# Add surrogate key
# dim_patient['member_sk'] = range(1, len(dim_patient) + 1)

# Hash patient_id for privacy
def hash_id(x):
    return hash(str(x))

dim_patient = df[df['age'] > 0].copy()
dim_patient['member_sk'] = range(1, len(dim_patient) + 1)
dim_patient['member_id_hashed'] = dim_patient['member_id'].apply(hash_id)


# dim_patient['patient_id_hashed'] = dim_patient['patient_id'].apply(hash_id)

# Select final columns
dim_patient = dim_patient[['member_sk', 'member_id_hashed', 'age', 'sex', 'region']]
dim_patient.to_csv('dim_patient.csv', index=False)
# Preview
print(dim_patient.head())
dim_patient.shape

   member_sk     member_id_hashed  age     sex     region
0          1 -3539596625162674163   63  Female  Northeast
1          2  1718106582376473760   23  Female    Midwest
2          3  6287834541517452339   66    Male  Northwest
3          4 -8192269875617132171   66    Male  Northwest
4          5 -8138215182438056560    4    Male  Northeast


(99029, 5)

In [74]:

# Fix column names based on your dataset
dim_provider = df[['provider_id', 'speciality', 'npi_number']].drop_duplicates().reset_index(drop=True)

# Create surrogate key
dim_provider['provider_sk'] = range(1, len(dim_provider) + 1)

# Handle missing values
dim_provider['provider_id'] = dim_provider['provider_id'].fillna(-1)
dim_provider['npi_number'] = dim_provider['npi_number'].fillna(-1)
dim_provider['speciality'] = dim_provider['speciality'].fillna("Unknown")

# Reorder columns
dim_provider = dim_provider[['provider_sk', 'provider_id', 'speciality', 'npi_number']]

# Add default Unknown provider if needed
default_provider_sk = 0
if -1 in dim_provider['provider_id'].values:
    unknown_row = pd.DataFrame(
        [[default_provider_sk, -1, 'Unknown', -1]],
        columns=['provider_sk', 'provider_id', 'speciality', 'npi_number']
    )
    dim_provider = pd.concat([unknown_row, dim_provider], ignore_index=True)

# Save
dim_provider.to_csv('dim_provider.csv', index=False)

print(dim_provider.head())
dim_provider.shape



   provider_sk provider_id         speciality  npi_number
0            1    PROV0166  Internal Medicine  9068671239
1            2    PROV0185        Orthopedics  6895786277
2            3    PROV0163        Orthopedics  5176466344
3            4    PROV0038         Pediatrics  1243376573
4            5    PROV0009  Internal Medicine  6827138318


(100000, 4)

In [75]:

total_rows = len(dim_provider['provider_id'])
unique_rows = len(dim_provider['provider_id'].unique())

print(f"Total rows: {total_rows}")
print(f"Unique provider IDs: {unique_rows}")

if unique_rows < total_rows:
    print(" There are duplicates.")
else:
    print(" No duplicates.")


Total rows: 100000
Unique provider IDs: 500
 There are duplicates.


In [78]:
import pandas as pd

# Load datasets
claims_df = pd.read_csv("synthetic_data.csv")
cpt_df = pd.read_csv("cpt4.csv")
icd_df = pd.read_csv("icd10_mapped_output.csv")

# --- STEP 0: Standardize procedure_code as STRING everywhere ---
claims_df['procedure_code'] = claims_df['procedure_code'].astype(str)
cpt_df['com.medigy.persist.reference.type.clincial.CPT.code'] = cpt_df['com.medigy.persist.reference.type.clincial.CPT.code'].astype(str)
icd_df['ICD-10 Code'] = icd_df['ICD-10 Code'].astype(str)

# --- Rename columns ---
cpt_df = cpt_df.rename(columns={
    'com.medigy.persist.reference.type.clincial.CPT.code': 'procedure_code',
    'label': 'cpt_description'
})
icd_df = icd_df.rename(columns={
    'ICD-10 Code': 'procedure_code',
    'ICD Description': 'icd_description'
})

# --- STEP 1: Extract procedure data ---
dim_procedure = claims_df[['procedure_code', 'procedure_description', 'category']].copy()

# --- STEP 2: Merge CPT descriptions ---
dim_procedure = dim_procedure.merge(
    cpt_df[['procedure_code', 'cpt_description']],
    on='procedure_code',
    how='left'
)

# --- STEP 3: Merge ICD descriptions ---
dim_procedure = dim_procedure.merge(
    icd_df[['procedure_code', 'icd_description']],
    on='procedure_code',
    how='left'
)

# --- STEP 4: Choose best description ---
dim_procedure['final_description'] = (
    dim_procedure['procedure_description']
        .fillna(dim_procedure['cpt_description'])
        .fillna(dim_procedure['icd_description'])
        .fillna("Unknown Procedure")
)

# --- STEP 5: Fill missing category ---
dim_procedure['category'] = dim_procedure['category'].fillna("Uncategorized")

# --- STEP 6: Remove duplicates ---
dim_procedure = dim_procedure[['procedure_code', 'final_description', 'category']].drop_duplicates()

# --- STEP 7: Create surrogate key ---
dim_procedure['proc_sk'] = range(1, len(dim_procedure) + 1)

# --- Final order ---
dim_procedure = dim_procedure[['proc_sk', 'procedure_code', 'final_description', 'category']]

# Save output
dim_procedure.to_csv("dim_procedure.csv", index=False)

dim_procedure.head()


,proc_sk,procedure_code,final_description,category
0,1,99214,"Office Visit, Est. Patient (moderate)",Specialty Care
1,2,90716,Tdap Vaccine,Primary Care
3,3,99214,"Office Visit, Est. Patient (moderate)",Primary Care
4,4,99283,Emergency Dept Visit,Primary Care
5,5,90471,Immunization Admin,Primary Care


In [79]:
print(cpt_df.columns)
print(icd_df.columns)


Index(['procedure_code', 'cpt_description'], dtype='object')
Index(['Diagnosis', 'procedure_code', 'icd_description', 'Similarity Score',
       'Justification', 'Alternative Suggestions', 'Needs Review'],
      dtype='object')


In [80]:

df.head()

,member_id,encounter_id,age,sex,weight,bmi,smoker,region,race,provider_id,...,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,claim_id,diag_1,diag_2,diag_3
0,MEMJTPNPBKRBD,ENC202308035284,63,Female,105.4,34.7,False,Northeast,White,PROV0166,...,No,No,No,Steady,Yes,>30,CLM63435S4SVP54OA,285.9,NaN,NaN
1,MEM2UCIN3S40P,ENC202204298091,23,Female,78.8,27.5,False,Midwest,AfricanAmerican,PROV0185,...,No,No,No,Down,Yes,NO,CLMEWR5HTDMRVBA0,786.50,NaN,NaN
2,MEMUP3B47V9G1,ENC202303201658,66,Male,90.6,28.2,False,Northwest,White,PROV0163,...,No,No,No,Steady,Yes,<30,CLMZLPJAB0F9Z1UH,285.9,401.9,NaN
3,MEMWZPL5NLUK4,ENC202202147829,66,Male,87.1,31.9,False,Northwest,AfricanAmerican,PROV0038,...,No,No,No,No,No,NO,CLMSFPKK9TFDOZ6,428.0,NaN,V70.0
4,MEMK0VU38S81E,ENC202207277691,4,Male,57.4,20.6,False,Northeast,Hispanic,PROV0009,...,Steady,Steady,Steady,Steady,No,<30,CLMSLRNZVLWC44GFS,272.4,496,NaN


In [81]:

# import pandas as pd

# # Assuming df is your main claims dataset
# # Load updated dataset with age categories
# df = pd.read_csv("synthetic_data_2025-12-05.csv")

# --- Create fact_claim ---
fact_claim = df[['member_id', 'provider_id', 'procedure_code', 'date_of_service',
                 'billed_amount', 'paid_amount', 'adjudication_status']].copy()

fact_claim['procedure_code'] = fact_claim['procedure_code'].astype(str)
dim_procedure['procedure_code'] = dim_procedure['procedure_code'].astype(str)


# Add surrogate key for fact table
fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)

fact_claim['member_id_hashed'] = fact_claim['member_id'].apply(hash_id)
patient_map = dict(zip(dim_patient['member_id_hashed'], dim_patient['member_sk']))
fact_claim['member_sk'] = fact_claim['member_id_hashed'].map(patient_map)


# Map provider_sk from dim_provider
provider_map = dict(zip(dim_provider['provider_id'], dim_provider['provider_sk']))
fact_claim['provider_sk'] = fact_claim['provider_id'].map(provider_map)

# Map proc_sk from dim_procedure
proc_map = dict(zip(dim_procedure['procedure_code'], dim_procedure['proc_sk']))
fact_claim['proc_sk'] = fact_claim['procedure_code'].map(proc_map)


# Drop original IDs for privacy concerns 
fact_claim = fact_claim[['claim_sk', 'member_sk', 'provider_sk', 'proc_sk',
                          'date_of_service', 'billed_amount', 'paid_amount', 'adjudication_status']]

# Validate
print(" fact_claim table created successfully!")
print(fact_claim.head())

# Save to CSV
fact_claim.to_csv('fact_claim.csv', index=False)


 fact_claim table created successfully!
   claim_sk  member_sk  provider_sk  proc_sk date_of_service  billed_amount  \
0         1    34260.0        99743       32      2022-07-29         410.56   
1         2    98549.0        99390       59      2022-06-29         304.94   
2         3    97145.0        98869       32      2023-06-24         254.28   
3         4    84068.0        99252       32      2022-12-06         467.48   
4         5    92133.0        99236       37      2022-10-31         413.19   

   paid_amount adjudication_status  
0       260.53            Approved  
1       276.43            Approved  
2       181.17            Approved  
3       390.19            Approved  
4       306.15            Approved  


In [82]:

# Add surrogate key first
fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)

# Now run validation
validation_results = []

# 1. Uniqueness Checks
checks = {
    'dim_patient.member_sk': dim_patient['member_sk'].is_unique,
    'dim_provider.provider_sk': dim_provider['provider_sk'].is_unique,
    'dim_procedure.proc_sk': dim_procedure['proc_sk'].is_unique,
    'fact_claim.claim_sk': fact_claim['claim_sk'].is_unique
}
for name, result in checks.items():
    validation_results.append({'Check': f'Unique {name}', 'Status': 'PASS' if result else 'FAIL'})

# 2. Null Checks in Critical Columns
critical_fact_cols = ['claim_sk', 'member_sk', 'provider_sk', 'proc_sk', 'date_of_service']
for col in critical_fact_cols:
    null_count = fact_claim[col].isnull().sum()
    validation_results.append({'Check': f'Nulls in fact_claim.{col}', 'Status': 'PASS' if null_count == 0 else f'FAIL ({null_count} nulls)'})

# 3. Referential Integrity Checks
invalid_patient_refs = fact_claim[~fact_claim['member_sk'].isin(dim_patient['member_sk'])]
validation_results.append({'Check': 'Referential integrity patient_sk', 'Status': 'PASS' if invalid_patient_refs.empty else f'FAIL ({len(invalid_patient_refs)} invalid)'})

invalid_provider_refs = fact_claim[~fact_claim['provider_sk'].isin(dim_provider['provider_sk'])]
validation_results.append({'Check': 'Referential integrity provider_sk', 'Status': 'PASS' if invalid_provider_refs.empty else f'FAIL ({len(invalid_provider_refs)} invalid)'})

invalid_proc_refs = fact_claim[~fact_claim['proc_sk'].isin(dim_procedure['proc_sk'])]
validation_results.append({'Check': 'Referential integrity proc_sk', 'Status': 'PASS' if invalid_proc_refs.empty else f'FAIL ({len(invalid_proc_refs)} invalid)'})

# Export validation report
report_df = pd.DataFrame(validation_results)
report_df.to_csv('validation_report.csv', index=False)

print(" Validation completed. Report saved as validation_report.csv")
print(report_df)


 Validation completed. Report saved as validation_report.csv
                                  Check             Status
0          Unique dim_patient.member_sk               PASS
1       Unique dim_provider.provider_sk               PASS
2          Unique dim_procedure.proc_sk               PASS
3            Unique fact_claim.claim_sk               PASS
4          Nulls in fact_claim.claim_sk               PASS
5         Nulls in fact_claim.member_sk    FAIL (11 nulls)
6       Nulls in fact_claim.provider_sk               PASS
7           Nulls in fact_claim.proc_sk               PASS
8   Nulls in fact_claim.date_of_service               PASS
9      Referential integrity patient_sk  FAIL (11 invalid)
10    Referential integrity provider_sk               PASS
11        Referential integrity proc_sk               PASS


In [83]:

import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load your DataFrames into SQLite
dim_patient.to_sql('dim_patient', conn, index=False)
dim_provider.to_sql('dim_provider', conn, index=False)
dim_procedure.to_sql('dim_procedure', conn, index=False)
fact_claim.to_sql('fact_claim', conn, index=False)

# 1. Top Providers by Paid Amount

query1 = """
SELECT p.provider_id, p.speciality, SUM(f.paid_amount) AS total_paid
FROM fact_claim f
JOIN dim_provider p ON f.provider_sk = p.provider_sk
GROUP BY p.provider_id, p.speciality
ORDER BY total_paid DESC
LIMIT 10;
"""
result1 = pd.read_sql_query(query1, conn)
print("\nTop Providers by Paid Amount:")
print(result1)






Top Providers by Paid Amount:
  provider_id         speciality  total_paid
0    PROV0267  Internal Medicine    65643.26
1    PROV0420         Cardiology    64657.86
2    PROV0345            Surgery    64272.31
3    PROV0209   General Practice    62775.31
4    PROV0073   General Practice    62594.79
5    PROV0200   General Practice    62091.52
6    PROV0013  Internal Medicine    62066.72
7    PROV0307  Internal Medicine    61440.84
8    PROV0078        Orthopedics    61354.81
9    PROV0079          Radiology    61325.05


In [84]:

import os
import shutil
from datetime import datetime

source_file = 'fact_claim.csv'
landing_zone = 'landing/fact_claim/'
staging_zone = 'staging/fact_claim/'

# Create directories if they don't exist
os.makedirs(landing_zone, exist_ok=True)
os.makedirs(staging_zone, exist_ok=True)

# Step 1: Move file to landing zone
shutil.copy(source_file, landing_zone)

# Step 2: Log metadata
log_file = 'ingestion_log.txt'
with open(log_file, 'a') as log:
    log.write(f"{source_file}, {datetime.now()}, {os.path.getsize(source_file)} bytes\n")

# Step 3: Validate and move to staging
if os.path.exists(os.path.join(landing_zone, source_file)):
    shutil.move(os.path.join(landing_zone, source_file), staging_zone)
    print("File moved to staging successfully!")


Error: Destination path 'staging/fact_claim/fact_claim.csv' already exists

In [85]:
print(os.listdir('staging/fact_claim'))

['fact_claim.csv']


In [87]:

import pandas as pd

# Load the unified dataset
unified_df = pd.read_csv('synthetic_data.csv')

# Select relevant columns for ingestion layer fact_claim
fact_claim_ingestion = unified_df[[
    'member_id', 'provider_id', 'region', 'procedure_code', 'procedure_description',
    'date_of_service', 'pre_adjudication_date', 'adjudication_date', 'payment_date',
    'adjudication_status', 'billed_amount', 'paid_amount','sex','age','region','weight',
]].copy()

# Rename columns to match fact_claim structure
fact_claim_ingestion.rename(columns={
    'member_id': 'member_sk',
    'provider_id': 'provider_sk'
}, inplace=True)

# Add claim_sk as a sequential ID
fact_claim_ingestion.insert(0, 'claim_sk', range(1, len(fact_claim_ingestion) + 1))

# Save the enriched ingestion layer table
fact_claim_ingestion.to_csv('fact_claim_ingestion.csv', index=False)

print("Ingestion layer fact_claim table created successfully with new columns:")
print(fact_claim_ingestion.head())
print(f"Total rows: {len(fact_claim_ingestion)}")


Ingestion layer fact_claim table created successfully with new columns:
   claim_sk      member_sk provider_sk     region  procedure_code  \
0         1  MEMJTPNPBKRBD    PROV0166  Northeast           99214   
1         2  MEM2UCIN3S40P    PROV0185    Midwest           90716   
2         3  MEMUP3B47V9G1    PROV0163  Northwest           99214   
3         4  MEMWZPL5NLUK4    PROV0038  Northwest           99214   
4         5  MEMK0VU38S81E    PROV0009  Northeast           99283   

                   procedure_description date_of_service  \
0  Office Visit, Est. Patient (moderate)      2022-07-29   
1                           Tdap Vaccine      2022-06-29   
2  Office Visit, Est. Patient (moderate)      2023-06-24   
3  Office Visit, Est. Patient (moderate)      2022-12-06   
4                   Emergency Dept Visit      2022-10-31   

  pre_adjudication_date adjudication_date payment_date adjudication_status  \
0            2022-08-03        2022-08-05   2022-09-04            Approved